LangGraph is a part of LangChain toolkit. It provides functionality for building and execution of  DAGs

[[guide]](https://langchain-ai.github.io/langgraph/)
[[reference]](https://langchain-ai.github.io/langgraph/reference/)

It's not much different from regular DAG execution engines, but its main focus is Language modeling pipelines (LLM completions, chatbots etc)



StateGraph represenets an execution graph

State = context that holds current variable values
Each node processes reads a State as input and returns State as output

DAG construction:
- add_node
- add_edge
- add_conditional_edges

compile()<br>it does not do much - it just evaluates that the graph is correct. Also you define checkpointing streategy. But after compilation you can manipulate the graph.

Running DAG:
- invoke()
- stream()

Node = a function that takes State as an argument returns dictionary of <br>
*it is ok to return only a subset of variables (for example, the changed ones)<br>
**if you return new non-existing variables, they will be ignored

Two constants represent starting and ending nodes (they do nothing)<br>
```from langgraph.graph import START, END```

#### Language models
Neither LangGraph nor LangChain work with LLM models, it just generates a corresponding API call. 

Interfaces the LangChain provides:
- LLM servers: vLLM, ollama
- HugginfacePipeline - imports the transformers library and uses its classes
- HugginfaceHub - sends requests to Hugginface cloud
- HugginfaceEndpoint - sends requests to a preconfigured server
- OpenAI, Antopic - API calls to OpenAI models
- ChatOpenAI, CHatAntropic - API calls to chat versions of models
- Google, Vertex

#### Typing

Due to specific purpose the library relies heavily on Python Typing capabilities. It helps define schema of the processed data.

Many classes (like State) are annotated dictionaries

You can annotate your types with metadata using Annotate class. In LangGraph you often annotate types with their aggregation mode:
- add
- values
- update
```python
from typing_extensions import Annotated

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add]
    extra_field: int
```

#### Graph States
Persistence layer = saves the state of the context (variable values) during execution
Superstep = execution iteration (parallel edges = same step, sequential = separate steps)
Thread = a way to logically separate runs (one thread for each run / multiple threads in one run / multiple runs in one thread)
Checpointing = saving current graph state
Several storage engines:
- InMemorySaver
- PostgresSaver

How you can interact with grah execution:
- get_state()
- get_state_history()
- invoke(checkpoint)<br>starts execution from paricular point
- update_state(val)<br>changes current variable values
- InMemoryStore


### Human-in-the-loop
Graphs can be paused to human intervention (if model decision is not compelling enough or before some critical action)

How this is done:
- call an __interrupt()__ function in the node code<br>it throws an exception, pauses the execution of the graph until the next Command() call<br>argument = value to evaluate by human
- in order to be able to continue the checkpointing must be enabled
- issue Command ```graph.invoke(Command(resume=val))``` to continue<br>it will restart the interrupted node ignoring interupt() function

- Send()<br>

## Tools

To enhance generation with` tool calls you need to register a list of available tools (tool = function names)
```
# augmenting LLMs with tool capabilities
llm.bind_tools([tool1, tool2])
```

```
# augmenting agents
agent = create_react_agent(model="anthropic:claude-3-7-sonnet",tools=[multiply])
```

Model will automatically generate calls if necessary (stored in tool_calls)
```
response_message = model_with_tools.invoke("what's 42 x 7?")
tool_call = response_message.tool_calls[0]
multiply.invoke(tool_call)
```


To create a custom tool annotate a function with tool

```
from langchain_core.tools import tool
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b
```

A node that executes an external tool
from langgraph.prebuilt import ToolNode
tool_node = ToolNode([tool_func])

There is a plethora of prebuilt 3rd party tools and integrations<br>Here is an example:
```
from langchain_tavily import TavilySearch
tool = TavilySearch(max_results=5, topic="general")
```
Direct invocation
```python
tool.invoke({'query':'search smth'})
```

ToolCall is a request generated by the model (contains more meta fields)
```python
tool_call = {...}
tool.invoke(tool_call)
```